# 04 — Fix Upstream Routing for Split Polygons

**Purpose:** After splitting HUC12 polygons (notebook 03), some upstream HUC12s
that nominally drain *into* a gauged HUC12 may not actually pass through the gauge.
This notebook identifies those cases using NLDI flowline tracing and re-routes
them to the `HUC_12 + "0"` remainder polygon, preserving hydrologic continuity.

**Logic:**  
For each gauged HUC12 with more than one upstream neighbor:
1. Fetch upstream main-stem and tributary flowlines via NLDI.
2. For each upstream HUC12 whose `HU_12_DS` points to the gauged polygon,
   check whether the HUC12 geometry **intersects** any upstream flowline.
3. If it does **not** intersect → it bypasses the gauge → update `HU_12_DS`
   to `gauged_HUC_12 + "0"` (the remainder polygon).

**Input:** `cache/nb03_intermediate.pkl`  
**Output:** `UMRB_HUC12_with_sites_modified.shp`

In [ ]:
# =============================================================================
# USER SETTINGS
# =============================================================================

WORKING_DIR = "/path/to/your/data"

# Output: final corrected HUC12 network shapefile
HUC12_FINAL_SHP = f"{WORKING_DIR}/UMRB_HUC12_with_sites_modified.shp"

# Must match CACHE_DIR used in notebooks 02 and 03
CACHE_DIR = "./cache"

# =============================================================================
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

sys.path.insert(0, '.')
from utils.nldi_helpers import fetch_all_upstream_flowlines

# Load intermediate outputs from notebook 03
with open(os.path.join(CACHE_DIR, 'nb03_intermediate.pkl'), 'rb') as f:
    nb03 = pickle.load(f)

UMRB_HUC12_with_sites     = nb03['UMRB_HUC12_with_sites']
UMRB_HUC12_with_sites_new = nb03['UMRB_HUC12_with_sites_new']
filtered_site_loc         = nb03['filtered_site_loc']
CACHE_DIR                 = nb03.get('CACHE_DIR', CACHE_DIR)

os.makedirs(os.path.dirname(HUC12_FINAL_SHP), exist_ok=True)
print(f'Network records: {len(UMRB_HUC12_with_sites_new)}')

## 1. Identify gauged HUC12s with multiple upstream neighbors

In [ ]:
gdf = UMRB_HUC12_with_sites_new.copy()

# Count how many HUC12s list each code as their downstream
upstream_counts = gdf['HU_12_DS'].value_counts()

valid_sites_gdf = gdf[~gdf['site_no'].isna()].copy()
valid_sites_gdf['upstream_count'] = valid_sites_gdf['HUC_12'].map(
    lambda x: upstream_counts.get(x, 0)
)

# Only these need flowline-based routing correction
multi_upstream = valid_sites_gdf[valid_sites_gdf['upstream_count'] > 1].copy()
print(f'Gauged HUC12s with multiple upstream neighbors: {len(multi_upstream)}')
print(multi_upstream[['HUC_12', 'site_no', 'upstream_count']])

## 2. Fetch upstream flowlines for affected gauge sites

In [ ]:
# Results are cached per-site under CACHE_DIR/nldi/
nldi_cache = os.path.join(CACHE_DIR, 'nldi')
Upstream_gpd = fetch_all_upstream_flowlines(
    list(multi_upstream['site_no'].unique()),
    cache_dir=nldi_cache
)
print(f'Total upstream flowline segments: {len(Upstream_gpd)}')

## 3. Re-route upstream HUC12s that do not intersect the gauge flowlines

In [ ]:
def reroute_non_contributing_upstreams(
    network_gdf: gpd.GeoDataFrame,
    multi_upstream_gdf: gpd.GeoDataFrame,
    upstream_flowlines: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    For each gauged HUC12 with multiple upstream neighbors, re-route any
    upstream HUC12 that does not geometrically intersect the gauge's upstream
    flowlines. Those polygons drain to the remainder part, not through the gauge.

    A non-intersecting upstream HUC12's ``HU_12_DS`` is changed from the gauged
    HUC12 code to ``gauged_code + '0'`` (the remainder polygon). If that code
    does not yet exist in the network (edge case), fall back to the gauged
    HUC12's own downstream code.

    Parameters
    ----------
    network_gdf : GeoDataFrame
        Full HUC12 network.
    multi_upstream_gdf : GeoDataFrame
        Gauged HUC12s with more than one upstream neighbor.
    upstream_flowlines : GeoDataFrame
        NLDI flowlines with a ``site_no`` column.

    Returns
    -------
    GeoDataFrame
        Network with corrected ``HU_12_DS`` values.
    """
    result = network_gdf.copy()

    for _, row in multi_upstream_gdf.iterrows():
        huc_12  = row['HUC_12']
        site_no = row['site_no']

        site_flowlines    = upstream_flowlines[upstream_flowlines['site_no'] == site_no]
        candidate_upstream = result[result['HU_12_DS'] == huc_12]

        for idx, upstream_row in candidate_upstream.iterrows():
            intersects = site_flowlines.geometry.intersects(upstream_row['geometry']).any()

            if not intersects:
                proposed_ds = f"{huc_12}0"

                if proposed_ds in result['HUC_12'].values:
                    result.at[idx, 'HU_12_DS'] = proposed_ds
                else:
                    # Fallback: route past the gauge to its own downstream
                    fallback = result.loc[result['HUC_12'] == huc_12, 'HU_12_DS'].values
                    if len(fallback) > 0:
                        result.at[idx, 'HU_12_DS'] = fallback[0]

    return result

In [ ]:
UMRB_HUC12_final = reroute_non_contributing_upstreams(
    UMRB_HUC12_with_sites_new,
    multi_upstream,
    Upstream_gpd,
)
print('Routing correction complete.')

## 4. Visual QC: routing before and after correction

In [ ]:
for _, mrow in multi_upstream.iterrows():
    huc12 = mrow['HUC_12']
    site  = mrow['site_no']

    upstream_orig = UMRB_HUC12_with_sites[UMRB_HUC12_with_sites['HU_12_DS'] == huc12]
    upstream_new  = UMRB_HUC12_final[UMRB_HUC12_final['HU_12_DS'] == huc12]
    part_a        = UMRB_HUC12_final[UMRB_HUC12_final['HUC_12'] == huc12]
    part_b        = UMRB_HUC12_final[UMRB_HUC12_final['HUC_12'] == f'{huc12}0']
    site_pt       = filtered_site_loc[filtered_site_loc['site_no'] == site]

    context_frames = [f for f in [upstream_orig, upstream_new, part_a, part_b] if not f.empty]
    bounds = pd.concat(context_frames).total_bounds

    fig, ax = plt.subplots(figsize=(9, 8))
    UMRB_HUC12_final.plot(ax=ax, color='white', edgecolor='lightgrey', linewidth=0.3)
    upstream_orig.plot(ax=ax, color='lightblue', edgecolor='black', alpha=0.5,
                       label='Upstream (original routing)')
    upstream_new.plot(ax=ax, color='steelblue', edgecolor='black', alpha=0.5,
                      label='Upstream (corrected routing)')
    part_a.plot(ax=ax, color='red', edgecolor='black', alpha=0.6, label='Inside gauge basin')
    if not part_b.empty:
        part_b.plot(ax=ax, color='orange', edgecolor='black', alpha=0.6,
                    label='Outside gauge basin (remainder)')
    site_pt.plot(ax=ax, color='yellow', markersize=60, zorder=5, label='Gauge site')

    site_flows = Upstream_gpd[Upstream_gpd['site_no'] == site]
    if not site_flows.empty:
        site_flows.plot(ax=ax, linewidth=0.8, color='purple', alpha=0.8,
                        label='Upstream flowlines')

    for _, lrow in pd.concat(context_frames).iterrows():
        pt = lrow['geometry'].representative_point()
        ax.text(pt.x, pt.y, lrow['HUC_12'], ha='center', fontsize=7, style='italic')

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_title(f'Routing correction: HUC12 {huc12} ({site})', fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(loc='upper right', fontsize=8)

    print(f'\nHUC12 {huc12} — upstreams after correction:')
    print(UMRB_HUC12_final[UMRB_HUC12_final['HU_12_DS'] == huc12]['HUC_12'].values)

    plt.tight_layout()
    plt.show()

## 5. Save final output

In [ ]:
UMRB_HUC12_final.to_file(HUC12_FINAL_SHP, driver='ESRI Shapefile')
print(f'Saved: {HUC12_FINAL_SHP}')
print(f'Total HUC12 records  : {len(UMRB_HUC12_final)}')
print(f'  Gauged             : {UMRB_HUC12_final["site_no"].notna().sum()}')
print(f'  Synthetic (split)  : {UMRB_HUC12_final["HUC_12"].astype(str).str.len().eq(13).sum()}')

## 6. Reload and verify

In [ ]:
verify = gpd.read_file(HUC12_FINAL_SHP)
print(f'Records reloaded: {len(verify)}')
print(verify[['HUC_12', 'HU_12_DS', 'site_no']].head(10))